# ABSA Phase 1 샌드박스 — JSON Schema + Boundary 강화 프롬프트

**목적**: v9 (Macro F1=0.7032)의 zero-sum 천장을 깰 수 있는지 **100건**으로 빠른 검증.

**격리 원칙**:
- `absa_v9.py` 수정 없음 — 헬퍼만 import (load_golden_set, build_batch_examples, evaluate)
- 검증셋 973건(Few-shot 27건 제외) 풀에서 100건 random sample (random_state=42)
- 본 서버 / 대시보드 영향 없음, 노트북 단독 실행

**Phase 1 변경 3가지**:
1. **고도화 System Prompt** — 핏/사이즈 vs 브랜드/헤리티지 boundary 명문화 (mutually exclusive 규칙)
2. **Ollama JSON Schema 강제** — `format='json'` + 6키 검증 → parsing fragility 제거
3. **즉석 Macro F1 평가** — v9 baseline(0.7032) 대비 Δ 출력

**예상 Uplift**: +0.04 ~ +0.06 (Conservative 시나리오) → 0.74~0.76 도달 가능성 검증


In [ ]:
# ── [1/6] 환경 + 데이터 로드 + 100건 샘플 ───────────────────
import os
import sys
import json
import time
from pathlib import Path

ABSA_DIR = Path("/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/absa_exaone")
os.chdir(ABSA_DIR)
if str(ABSA_DIR) not in sys.path:
    sys.path.insert(0, str(ABSA_DIR))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# v9 헬퍼 재사용 (수정 없음)
from absa_v9 import (
    ASPECTS, LABELS,
    load_golden_set, build_few_shot_examples, build_batch_examples,
    evaluate,
)

SEED = 42
N_SAMPLES = 100
GOLDEN_PATH = "absa_golden_set_1000_v23.xlsx"

golden = load_golden_set(GOLDEN_PATH)

# v9와 동일 방식으로 Few-shot 27건 식별 → 검증 풀에서 제외 (data leakage 방지)
few_shots = build_few_shot_examples(golden, n_per_class=3, seed=SEED)
fs_contents = {ex_c for fs in few_shots.values() for ex_c, _ in fs}
val_pool = golden[~golden["content_clean"].isin(fs_contents)].copy()

sample = val_pool.sample(n=N_SAMPLES, random_state=SEED).reset_index(drop=True)

print(f"골든셋 전체   : {len(golden):,}건")
print(f"검증 풀(FS 제외): {len(val_pool):,}건")
print(f"샘플          : {len(sample):,}건")
print()
print("[샘플 분포 — 6속성 × P/N/X]")
for asp in ASPECTS:
    c = sample[asp].value_counts().to_dict()
    print(f"  {asp:>14}: P={c.get('P',0):>3} / N={c.get('N',0):>3} / X={c.get('X',0):>3}")


In [ ]:
# ── [2/6] Phase 1 고도화 System Prompt + JSON 출력 빌더 ─────
# v9 대비 변경:
#  - 6속성 mutually exclusive 정의 (어떤 속성이 어디 귀속되는지)
#  - "편하다/좋다/찰떡" 단독 → 핏/브랜드 모두 X
#  - "또 살게요/추천" → 브랜드 P, 핏 X (단순 상품 칭찬과 분리)
#  - 부정+긍정 혼재 시 후미 절 우선
#  - 출력 형식: JSON 강제 (코드블록·여분 텍스트 금지)

SYSTEM_BLOCK = '''한국어 애슬레저 리뷰를 6개 속성에 대해 P/N/X로 분류한다.
P=긍정 언급, N=부정 언급, X=해당 속성이 전혀 언급되지 않음.

[속성 정의 — 상호 배타]
1. 핏/사이즈      : 사이즈 수치, 기장, 허리, 어깨, 라인, Y존, 압박감, 라이즈
2. 소재/내구성    : 원단, 촉감, 두께, 보풀, 변색, 비침, 늘어남
3. 기능성         : 신축성, 통기성, 흡습/땀, 운동 활동성, 보온/냉감
4. 디자인         : 색상, 패턴, 실루엣, 외관, 코디 어울림
5. 브랜드/헤리티지 : 재구매 의사, 브랜드 충성·추천, 강추, 단골, 브랜드명 직접 언급
6. 가격/가치      : 가격 만족, 가성비, 할인, 비쌈, 저렴

[Boundary 규칙 — v9에서 헷갈리던 case 명시]
- "편해요/좋아요/찰떡/딱이에요/가볍다" 단독은 모든 속성 X
  (핏 직접 언급 없으면 핏=X, 브랜드 칭찬 없으면 브랜드=X)
- "예쁘다/이쁘다" 단독은 디자인=P, 다른 속성=X
- "또 살게요/계속 쓸 듯/추천/강추/단골/믿고 산다" → 브랜드=P
  (단순 상품 칭찬 "예쁘다·편하다"는 브랜드=X)
- "사이즈 작아요/한 치수 크다/허리 말림/기장 길다" → 핏=N (브랜드=X)
- 부정+긍정 혼재 시 후미 절(끝부분)이 부정이면 그 속성은 N
  예) "조이지 않고 잘 맞긴 한데 사이즈가 작다" → 핏=N
- 의류 명사(크롭/양면/셋업)는 옵션 설명일 뿐 핏/디자인 평가 아님 → X
- 가격 수치 단순 언급("6만원")은 가격=X. "이 가격에 가성비"만 가격=P

[출력 — 반드시 JSON, 코드블록·설명 금지]
{
  "핏/사이즈": "P|N|X",
  "소재/내구성": "P|N|X",
  "기능성": "P|N|X",
  "디자인": "P|N|X",
  "브랜드/헤리티지": "P|N|X",
  "가격/가치": "P|N|X"
}'''


def build_phase1_prompt(content: str, examples: list[tuple[str, dict[str, str]]]) -> str:
    """Phase 1 프롬프트 — JSON 강제 + boundary 명시."""
    parts = [SYSTEM_BLOCK, ""]
    for ex_content, ex_labels in examples:
        ex_json = json.dumps(
            {asp: ex_labels.get(asp, "X") for asp in ASPECTS},
            ensure_ascii=False,
        )
        parts.append(f'리뷰: "{ex_content}"')
        parts.append(f"출력: {ex_json}")
        parts.append("")
    parts.append(f'리뷰: "{content}"')
    parts.append("출력:")
    return "\n".join(parts)


# Few-shot 6건 (v9와 동일 방식)
batch_examples = build_batch_examples(golden, n_examples=6, seed=SEED)
print(f"Few-shot 예시: {len(batch_examples)}건")
print()
print("─ 프롬프트 미리보기 (앞 1500자) ─")
print(build_phase1_prompt(sample.iloc[0]["content_clean"], batch_examples)[:1500])


In [ ]:
# ── [3/6] Ollama JSON Schema 강제 호출 ───────────────────────
import ollama

MODEL = "exaone3.5:7.8b"
TEMPERATURE = 0.1
MAX_TOKENS = 200

# Ollama 0.5+ structured outputs (JSON Schema)
JSON_SCHEMA = {
    "type": "object",
    "properties": {asp: {"type": "string", "enum": LABELS} for asp in ASPECTS},
    "required": list(ASPECTS),
}

OLLAMA_OPTS = {"temperature": TEMPERATURE, "num_predict": MAX_TOKENS}


def _ollama_call(prompt: str, fmt):
    return ollama.generate(model=MODEL, prompt=prompt, format=fmt, options=OLLAMA_OPTS)


def call_exaone_json(prompt: str, retries: int = 3) -> dict[str, str]:
    """JSON Schema 우선 시도 → 실패 시 'json' 문자열로 폴백 + 재시도."""
    fallback = {asp: "X" for asp in ASPECTS}

    for attempt in range(retries):
        # 1차: structured outputs (Ollama 0.5+ / ollama-python 0.4+)
        # 2차: format='json' 문자열만 (구버전 호환)
        for fmt in (JSON_SCHEMA, "json"):
            try:
                resp = _ollama_call(prompt, fmt)
                text = resp["response"].strip()
                parsed = json.loads(text)

                result = {}
                for asp in ASPECTS:
                    v = str(parsed.get(asp, "X")).strip().upper()
                    result[asp] = v if v in LABELS else "X"
                return result
            except (json.JSONDecodeError, KeyError):
                continue  # JSON 안 옴 → 다음 fmt 시도
            except Exception as e:
                # ollama 서버 오류 — 같은 fmt로 retry는 안 하고 다음 fmt 시도
                last_err = e
                continue

        # 두 fmt 다 실패 → retry
        if attempt < retries - 1:
            time.sleep(1.5)

    print(f"[fail after {retries} retries] → fallback (all X)")
    return fallback


# Smoke test (1건)
test_content = sample.iloc[0]["content_clean"]
test_prompt = build_phase1_prompt(test_content, batch_examples)
test_pred = call_exaone_json(test_prompt)
print(f"테스트 리뷰: {str(test_content)[:80]}...")
print(f"테스트 예측: {test_pred}")
print()
gold_str = ", ".join(f"{a!r}: {sample.iloc[0][a]!r}" for a in ASPECTS)
print(f"Gold       : {{ {gold_str} }}")


In [ ]:
# ── [4/6] 100건 추론 실행 (단일 스레드, 진행률 표시) ────────────
contents = sample["content_clean"].tolist()
preds: list[dict[str, str]] = []

start = time.time()
for content in tqdm(contents, desc="Phase 1 샌드박스 추론"):
    prompt = build_phase1_prompt(str(content), batch_examples)
    preds.append(call_exaone_json(prompt))
elapsed = time.time() - start

print(f"\n완료: {len(preds)}건 / {elapsed:.1f}초 / {elapsed/len(preds):.2f}초/건")
print(f"v9 baseline: 8.62초/건 (workers=2 기준) — 단일 스레드라 순수 비교는 부적절")


In [ ]:
# ── [5/6] Macro F1 평가 + v9 대비 Δ ────────────────────────────
pred_df = sample[["sample_idx", "content_clean"]].copy()
for asp in ASPECTS:
    pred_df[asp] = [p[asp] for p in preds]

result = evaluate(pred_df, sample, join_key="sample_idx")

V9_BASELINE = 0.7032
delta = result["macro_f1"] - V9_BASELINE
sign = "+" if delta >= 0 else ""

print("=" * 64)
print(f"  Phase 1 샌드박스 (n={N_SAMPLES})  Macro F1 = {result['macro_f1']:.4f}")
print(f"  v9 baseline (n=973)              Macro F1 = {V9_BASELINE:.4f}")
print(f"  Δ                                          {sign}{delta:.4f}")
print("=" * 64)
print()

V9_PER_ASPECT = {
    "핏/사이즈": 0.6107, "소재/내구성": 0.7831, "기능성": 0.7322,
    "디자인": 0.7432, "브랜드/헤리티지": 0.6019, "가격/가치": 0.7478,
}
print("[속성별 F1] (P1=phase1_100, V9=v9_973)")
for asp in ASPECTS:
    p1 = result["per_aspect_f1"][asp]
    v9 = V9_PER_ASPECT[asp]
    d = p1 - v9
    flag = "✅" if d >= 0.02 else ("≈" if abs(d) < 0.02 else "❌")
    print(f"  {asp:>14}  P1={p1:.4f}  V9={v9:.4f}  Δ={d:+.4f}  {flag}")


In [ ]:
# ── [6/6] 핵심 오분류 진단 — 핏/사이즈, 브랜드/헤리티지 ────────
print("[Confusion — 핏/사이즈]")
print(result["confusion"]["핏/사이즈"])
print()
print("[Confusion — 브랜드/헤리티지]")
print(result["confusion"]["브랜드/헤리티지"])

# 핏 X→P 오분류 케이스 (v9 주요 병목)
m = pred_df.merge(sample, on="sample_idx", suffixes=("_pred", "_gold"))

print("\n— 핏/사이즈  gold=X → pred=P  (Phase 1 잔여 오분류) —")
mask = (m["핏/사이즈_gold"] == "X") & (m["핏/사이즈_pred"] == "P")
for _, r in m[mask].head(8).iterrows():
    print(f"  · {str(r['content_clean_gold'])[:80]}")

print("\n— 브랜드/헤리티지  gold=X → pred=P  (P-bias 잔여) —")
mask = (m["브랜드/헤리티지_gold"] == "X") & (m["브랜드/헤리티지_pred"] == "P")
for _, r in m[mask].head(8).iterrows():
    print(f"  · {str(r['content_clean_gold'])[:80]}")

print("\n— 브랜드/헤리티지  gold=P → pred=X  (P recall 누락) —")
mask = (m["브랜드/헤리티지_gold"] == "P") & (m["브랜드/헤리티지_pred"] == "X")
for _, r in m[mask].head(8).iterrows():
    print(f"  · {str(r['content_clean_gold'])[:80]}")
